# Hidden Markov Model from Scratch

- Gaussian emission model + Observable-factor Markov chain baseline
- Forward Algorithm
- Backward Algorithm
- Viterbi Algorithm
- Baum-Welch EM
- Regime analysis & interactive visualisations

Charts are interactive **Plotly** figures.

---

### References
1. Rabiner (1989) — *A tutorial on hidden Markov models*
2. Nguyen & Nguyen (2015) — *Hidden Markov Model for Stock Selection*, Risks 3(4)
3. McGreevy, J. (2021) — *Hidden Markov Models in Finance*, Imperial College MSc
4. Paolucci, R. (2025) — *HMMs for Quantitative Finance*, Quant Guild (reference PDF)
5. Chen, X. (2025) — *HMM-based market regime detection with RL for portfolio management*, IDS

---

## 0. Setup & Imports

In [1]:
%pip install -q numpy pandas scipy plotly yfinance ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import norm
import warnings
warnings.filterwarnings("ignore")

# Project utilities
from utils.data_utils import prepare_ticker_data, get_observation_sequence, time_split
from utils.viz_utils  import (plot_price_with_regimes, plot_regime_distributions,
                               plot_transition_matrix, plot_alpha_beta, plot_gamma,
                               plot_log_likelihood, plot_viterbi_path, plot_regime_stats)
from utils.metrics    import (regime_statistics, regime_persistence_summary,
                               conditional_sharpe, pairwise_wasserstein, aic, bic, hmm_n_params)

# Our from-scratch implementation
from hmm_core import GaussianHMM, forward_pass, backward_pass, viterbi, compute_gamma, compute_xi

print("All imports OK")

All imports OK


## 1. Data - S&P 500 (SPY) Daily Log-Returns

We fetch daily OHLCV data and compute:
$$r_t = \ln\!\left(\frac{P_t}{P_{t-1}}\right)$$
$$\hat{\sigma}^{(w)}_t = \sqrt{252}\cdot\text{std}_{\text{roll-}w}(r_t)$$

In [2]:
TICKER = "SPY"
df = prepare_ticker_data(TICKER, start="2005-01-01")
df_train, df_test = time_split(df, train_frac=0.80)

X_train = get_observation_sequence(df_train)
X_test  = get_observation_sequence(df_test)
X_all   = get_observation_sequence(df)

print(f"Total observations : {len(X_all)}")
print(f"Train             : {len(X_train)}  ({df_train.index[0].date()} – {df_train.index[-1].date()})")
print(f"Test              : {len(X_test)}   ({df_test.index[0].date()} – {df_test.index[-1].date()})")
print(f"\nReturn stats: mean={X_all.mean()*252:.2%}  ann.vol={X_all.std()*np.sqrt(252):.2%}")

Total observations : 5237
Train             : 4189  (2005-06-23 – 2022-02-10)
Test              : 1048   (2022-02-11 – 2026-04-17)

Return stats: mean=10.34%  ann.vol=19.16%


In [3]:
# Interactive price + realized volatility chart
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    row_heights=[0.5, 0.25, 0.25],
                    subplot_titles=("SPY Adjusted Close", "Daily Log-Return", "20-Day Realised Vol (ann.)"))

fig.add_trace(go.Scatter(x=df.index, y=df["Close"],
    line=dict(color="rgba(150,180,255,0.9)", width=1.2), name="Close"), row=1, col=1)

fig.add_trace(go.Bar(x=df.index, y=df["Returns"]*100,
    marker_color=np.where(df["Returns"] >= 0, "rgba(99,200,140,0.7)", "rgba(240,80,80,0.7)"),
    name="Return (%)"), row=2, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df["RVol_20d"]*100,
    line=dict(color="rgba(255,195,55,0.9)", width=1), name="RVol 20d (%)"), row=3, col=1)

fig.update_layout(title="SPY — Returns & Volatility Dynamics",
                  height=700, plot_bgcolor="rgba(15,17,22,1)",
                  paper_bgcolor="rgba(15,17,22,1)",
                  font=dict(color="#e0e0e0"), showlegend=True,
                  xaxis3_rangeslider_visible=False)
fig.update_xaxes(showgrid=True, gridcolor="rgba(80,80,80,0.3)")
fig.update_yaxes(showgrid=True, gridcolor="rgba(80,80,80,0.3)")
fig.show()

## 2. Observable-Factor Markov Chain Baseline

Before HMMs, we build the simpler **observable-factor Markov chain** from §2 of the reference PDF.

**Regime definition via quantile bucketing** (Eq. 10–12):
$$\mathcal{R}_{\text{Low}}  = \{t : \hat{\sigma}_t \leq Q_{0.33}\}$$
$$\mathcal{R}_{\text{Med}}  = \{t : Q_{0.33} < \hat{\sigma}_t \leq Q_{0.66}\}$$
$$\mathcal{R}_{\text{High}} = \{t : \hat{\sigma}_t > Q_{0.66}\}$$

**MLE transition matrix** (Eq. 13):
$$\hat{a}_{ij} = \frac{n_{ij}}{\sum_k n_{ik}}$$

In [4]:
from utils.data_utils import empirical_transition_matrix

# Observable labels already in df (from prepare_ticker_data)
vol_labels = df_train["VolRegime"].values
A_obs, states = empirical_transition_matrix(vol_labels, ["Low","Med","High"])

print("MLE Transition Matrix (Observable-Factor Markov Chain):")
print(pd.DataFrame(A_obs, index=states, columns=states).round(4))

fig = plot_transition_matrix(A_obs, state_names=states, 
                              title="Observable-Factor MC : Transition Matrix")
fig.show()

MLE Transition Matrix (Observable-Factor Markov Chain):
         Low     Med    High
Low   0.9421  0.0566  0.0013
Med   0.0674  0.8973  0.0353
High  0.0000  0.0326  0.9674


In [5]:
# Regime statistics for observable MC
ret_arr  = df_train["Returns"].values
lab_arr  = df_train["VolRegime"].values

stats_df = regime_statistics(ret_arr, lab_arr)
print(stats_df.to_string(index=False))

fig = plot_regime_distributions(df_train, "VolRegime", 
                                 title="Observable MC — Regime Return Distributions")
fig.show()

State  Mean Return (ann.)  Volatility (ann.)  Sharpe (ann.)  Skewness  Excess Kurtosis  # Days  % of Sample
 High            0.051595           0.300242       0.171846 -0.248662         6.632407    1411    33.683457
  Low            0.157882           0.083125       1.899325 -0.294233         1.030926    1502    35.855813
  Med            0.079365           0.130509       0.608123 -0.496005         0.967506    1276    30.460730


In [6]:
fig = plot_price_with_regimes(df_train, "VolRegime",
                              title="SPY — Observable Volatility Regimes")
fig.show()

## 3. Forward Algorithm

The **forward variable** $\alpha_t(i)$ is the joint probability of observing the sequence up to time $t$ **and** being in state $S_i$ at time $t$:
$$\alpha_t(i) = P(O_1, O_2, \ldots, O_t,\, Q_t = S_i \mid \lambda)$$

### 3.1 Initialisation (Eq. 18)
$$\alpha_1(i) = \pi_i \cdot b_i(O_1), \quad i = 1,\ldots,N$$

### 3.2 Recursion (Eq. 19)
$$\alpha_{t+1}(j) = \left[\sum_{i=1}^{N} \alpha_t(i)\, a_{ij}\right] b_j(O_{t+1}), \quad t = 1,\ldots,T-1$$

### 3.3 Termination (Eq. 20)
$$P(O \mid \lambda) = \sum_{i=1}^{N} \alpha_T(i)$$

Computational complexity: $O(N^2 T)$ — polynomial in $T$, not exponential.

> **Implementation note:** we use *scaled* forward variables to prevent numerical underflow.  
> $\ln P(O|\lambda) = \sum_t \ln c_t$ where $c_t$ are the per-step normalisation constants.

In [9]:
# Quick demonstration with manually set parameters (before training)
N_demo = 3
pi_demo = np.array([0.5, 0.3, 0.2])
A_demo  = np.array([[0.7, 0.2, 0.1],
                     [0.2, 0.6, 0.2],
                     [0.1, 0.3, 0.6]])
mu_demo    = np.array([-0.002, 0.001, -0.005])
sigma_demo = np.array([0.005,  0.012,  0.025])

alpha_demo, scales_demo = forward_pass(X_train[:50], pi_demo, A_demo, mu_demo, sigma_demo)

print(f"alpha shape  : {alpha_demo.shape}  (T=50, N=3)")
print(f"alpha[0]     : {alpha_demo[0].round(6)}  (sums to {alpha_demo[0].sum():.4f})")
print(f"alpha[-1]    : {alpha_demo[-1].round(6)}")
print(f"Log-likelihood (demo): {np.log(scales_demo).sum():.2f}")

alpha shape  : (50, 3)  (T=50, N=3)
alpha[0]     : [0.216703 0.470461 0.312835]  (sums to 1.0000)
alpha[-1]    : [0.430542 0.449532 0.119925]
Log-likelihood (demo): 172.75


## 4. Backward Algorithm

The **backward variable** $\beta_t(i)$ is the conditional probability of the *future* observations given state $S_i$ at time $t$:
$$\beta_t(i) = P(O_{t+1}, O_{t+2}, \ldots, O_T \mid Q_t = S_i,\, \lambda)$$

### 4.1 Initialisation (Eq. 22)
$$\beta_T(i) = 1, \quad i = 1,\ldots,N$$

### 4.2 Recursion (Eq. 23)
$$\beta_t(i) = \sum_{j=1}^{N} a_{ij}\, b_j(O_{t+1})\, \beta_{t+1}(j), \quad t = T-1,\ldots,1$$

### 4.3 Key identities (Eqs. 24–26)
$$P(O\mid\lambda) = \sum_{i=1}^N \alpha_t(i)\,\beta_t(i) \quad \text{(at any } t \text{)}$$
$$\gamma_t(i) = P(Q_t=S_i \mid O,\lambda) = \frac{\alpha_t(i)\,\beta_t(i)}{\sum_k \alpha_t(k)\,\beta_t(k)}$$
$$\xi_t(i,j) = P(Q_t=S_i,\, Q_{t+1}=S_j \mid O,\lambda) = \frac{\alpha_t(i)\,a_{ij}\,b_j(O_{t+1})\,\beta_{t+1}(j)}{\sum_{i'}\sum_{j'} \alpha_t(i')\,a_{i'j'}\,b_{j'}(O_{t+1})\,\beta_{t+1}(j')}$$

In [10]:
beta_demo = backward_pass(X_train[:50], A_demo, mu_demo, sigma_demo, scales_demo)
gamma_demo = compute_gamma(alpha_demo, beta_demo)

print(f"beta shape   : {beta_demo.shape}")
print(f"beta[-1]     : {beta_demo[-1].round(6)}  (initialised to 1 then scaled)")
print(f"gamma shape  : {gamma_demo.shape}")
print(f"gamma[0]     : {gamma_demo[0].round(4)}  (sums to {gamma_demo[0].sum():.4f})")

# Verify identity: γ sums to 1 at every t
assert np.allclose(gamma_demo.sum(axis=1), 1.0, atol=1e-8), "γ should sum to 1!"
print("\n γt(i) sums to 1 for all t — identity verified")

beta shape   : (50, 3)
beta[-1]     : [0.026861 0.026861 0.026861]  (initialised to 1 then scaled)
gamma shape  : (50, 3)
gamma[0]     : [0.3473 0.4512 0.2015]  (sums to 1.0000)

 γt(i) sums to 1 for all t — identity verified


In [11]:
fig = plot_alpha_beta(alpha_demo, beta_demo, max_t=50,
                      state_names=["S1","S2","S3"])
fig.show()

## 5. Viterbi Algorithm

Find the **most-probable hidden state sequence** $Q^* = \arg\max_Q P(Q \mid O, \lambda)$.

Define the Viterbi variable:
$$\delta_t(i) = \max_{Q_1,\ldots,Q_{t-1}} P(Q_1,\ldots,Q_t=S_i,\, O_1,\ldots,O_t \mid \lambda)$$

**Initialisation:**
$$\delta_1(i) = \pi_i \cdot b_i(O_1)$$

**Recursion** (in log-space to prevent underflow):
$$\log\delta_{t+1}(j) = \max_{i}\left[\log\delta_t(i) + \log a_{ij}\right] + \log b_j(O_{t+1})$$

Back-pointer:
$$\psi_t(j) = \arg\max_i\left[\delta_{t-1}(i)\cdot a_{ij}\right]$$

**Back-tracking** from $q_T^* = \arg\max_i \delta_T(i)$.

In [12]:
# Viterbi on demo parameters
path_demo, log_prob_demo = viterbi(X_train[:100], pi_demo, A_demo, mu_demo, sigma_demo)
print(f"Viterbi path (first 20): {path_demo[:20]}")
print(f"Log-probability of best path: {log_prob_demo:.2f}")

Viterbi path (first 20): [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Log-probability of best path: 315.73


## 6. Baum-Welch EM

The **Baum-Welch algorithm** is EM applied to HMMs:
$$\hat{\lambda} = \arg\max_\lambda P(O \mid \lambda)$$

### E-Step
Using current $\lambda^{(\text{old})}$, compute $\alpha_t(i)$, $\beta_t(i)$, then:
$$\gamma_t(i) = \frac{\alpha_t(i)\,\beta_t(i)}{\sum_k \alpha_t(k)\,\beta_t(k)}$$
$$\xi_t(i,j) = \frac{\alpha_t(i)\,a_{ij}\,b_j(O_{t+1})\,\beta_{t+1}(j)}{\sum_{i'}\sum_{j'}(\ldots)}$$

### M-Step — closed-form updates
$$\pi_i^{\text{new}} = \gamma_1(i) \tag{Eq. 27}$$
$$a_{ij}^{\text{new}} = \frac{\sum_{t=1}^{T-1} \xi_t(i,j)}{\sum_{t=1}^{T-1} \gamma_t(i)} \tag{Eq. 28}$$
$$\mu_j^{\text{new}} = \frac{\sum_t \gamma_t(j)\,O_t}{\sum_t \gamma_t(j)} \tag{Eq. 29}$$
$$\sigma_j^{2,\text{new}} = \frac{\sum_t \gamma_t(j)\,(O_t - \mu_j^{\text{new}})^2}{\sum_t \gamma_t(j)} \tag{Eq. 30}$$

**Convergence:** stop when $|\mathcal{L}^{(k)} - \mathcal{L}^{(k-1)}| < \varepsilon$ (Eq. 32).

> **Proposition 5.1 (Monotone improvement):** each EM iteration guarantees $\mathcal{L}^{(k+1)} \geq \mathcal{L}^{(k)}$.

In [13]:
# Train 3-state HMM from scratch
hmm3 = GaussianHMM(n_states=3, n_iter=300, tol=1e-7, random_state=42)
print("Training 3-state Gaussian HMM on SPY daily returns …")
hmm3.fit(X_train)
print()
print(hmm3.summary())

Training 3-state Gaussian HMM on SPY daily returns …
  Converged at iteration 125  (ΔLL = 9.00e-08)

GaussianHMM  N=3
  π  = [0. 1. 0.]
  A  =
       [9.688e-01 3.060e-02 6.000e-04]
       [0.0397 0.956  0.0043]
       [0.     0.0356 0.9644]
  µ  = [ 0.00119  -0.000105 -0.003313]
  σ  = [0.005091 0.012271 0.035446]
  Final LL = 13865.4278


In [14]:
fig = plot_log_likelihood(hmm3.log_likelihoods_,
                          title="Baum-Welch Convergence — 3-State Gaussian HMM on SPY")
fig.show()

## 7. Regime Analysis & Results

In [15]:
# Soft assignments (γ) and hard assignments
gamma_train = hmm3.predict_proba(X_train)
labels_soft = hmm3.predict(X_train)
labels_hard, log_p_viterbi = hmm3.predict_viterbi(X_train)

state_names = ["Low-Vol", "Med-Vol", "High-Vol"]

print("Soft (argmax γ) state counts:", dict(zip(*np.unique(labels_soft, return_counts=True))))
print("Viterbi state counts        :", dict(zip(*np.unique(labels_hard, return_counts=True))))

Soft (argmax γ) state counts: {np.int64(0): np.int64(2240), np.int64(1): np.int64(1719), np.int64(2): np.int64(230)}
Viterbi state counts        : {np.int64(0): np.int64(2284), np.int64(1): np.int64(1690), np.int64(2): np.int64(215)}


In [16]:
# State occupancy heat map over time
fig = plot_gamma(gamma_train, dates=df_train.index, state_names=state_names)
fig.show()

In [17]:
# Assign regime labels to df_train
df_train = df_train.copy()
df_train["HMM_State"] = labels_soft

fig = plot_price_with_regimes(df_train, "HMM_State",
                               title="SPY — 3-State HMM Regimes (Soft Assignment)")
fig.show()

In [18]:
fig = plot_regime_distributions(df_train, "HMM_State",
                                 title="HMM — Regime-Conditional Return Distributions")
fig.show()

In [19]:
fig = plot_transition_matrix(hmm3.A_, state_names=state_names,
                             title="Learned Transition Matrix A")
fig.show()

In [20]:
# Soft vs Viterbi comparison
df_vit = df_train.copy()
df_vit["Viterbi_State"] = labels_hard

fig = plot_viterbi_path(df_vit, labels_hard, state_names=state_names,
                         title="Viterbi Decoded Path — SPY")
fig.show()

In [21]:
# Regime statistics
stats_hmm = regime_statistics(df_train["Returns"].values, labels_soft)
print(stats_hmm.round(4).to_string(index=False))

 State  Mean Return (ann.)  Volatility (ann.)  Sharpe (ann.)  Skewness  Excess Kurtosis  # Days  % of Sample
     0              0.2953             0.0802         3.6806   -0.0310           0.4782    2240      53.4734
     1             -0.0186             0.1958        -0.0951   -0.1281           0.0457    1719      41.0360
     2             -0.9491             0.5807        -1.6344    0.1295           1.0013     230       5.4906


In [22]:
persistence = regime_persistence_summary(labels_soft)
print(persistence.to_string(index=False))

 State  Mean Duration (days)  Median Duration (days)  Max Duration (days)  # Episodes
     0             45.714286                    35.0                  306          49
     1             29.637931                    17.5                  226          58
     2             25.555556                     9.0                   75           9


In [23]:
sharpes = conditional_sharpe(df_train["Returns"].values, labels_soft)
print("Conditional annualised Sharpe per state:")
for s, sh in sharpes.items():
    print(f"  State {s} ({state_names[s]}): {sh:.3f}")

Conditional annualised Sharpe per state:
  State 0 (Low-Vol): 3.681
  State 1 (Med-Vol): -0.095
  State 2 (High-Vol): -1.634


In [24]:
W_dist = pairwise_wasserstein(df_train["Returns"].values, labels_soft)
print("\nPairwise Wasserstein-1 distances between regime distributions:")
print(W_dist.round(6))


Pairwise Wasserstein-1 distances between regime distributions:
          0         1         2
0  0.000000  0.005920  0.024688
1  0.005920  0.000000  0.018777
2  0.024688  0.018777  0.000000


## 8. Model Selection : Number of States N

Fit HMMs with N = 2, 3, 4, 5 states. Compare via AIC, BIC, and out-of-sample log-likelihood.

$$\text{AIC} = 2k - 2\mathcal{L}, \quad \text{BIC} = k \ln T - 2\mathcal{L}$$
where $k = N^2 + 2N$ is the number of free parameters.

In [26]:
results = []
hmm_models = {}

for N in range(2, 6):
    print(f"  Fitting {N}-state HMM …", end=" ")
    h = GaussianHMM(n_states=N, n_iter=300, tol=1e-7, random_state=42)
    h.fit(X_train)
    ll_train = h.log_likelihood(X_train)
    ll_test  = h.log_likelihood(X_test)
    k        = hmm_n_params(N)
    results.append({
        "N States": N,
        "Train LL":  round(ll_train, 1),
        "Test LL":   round(ll_test, 1),
        "AIC":       round(aic(ll_train, k), 1),
        "BIC":       round(bic(ll_train, k, len(X_train)), 1),
        "# Params":  k,
    })
    hmm_models[N] = h
    print(f"LL={ll_train:.1f}  AIC={aic(ll_train,k):.1f}  BIC={bic(ll_train,k,len(X_train)):.1f}")

print()
print(pd.DataFrame(results).to_string(index=False))

  Fitting 2-state HMM …   Converged at iteration 90  (ΔLL = 6.52e-08)
LL=13609.3  AIC=-27202.6  BIC=-27151.9
  Fitting 3-state HMM …   Converged at iteration 125  (ΔLL = 9.00e-08)
LL=13865.4  AIC=-27700.9  BIC=-27605.8
  Fitting 4-state HMM …   Converged at iteration 177  (ΔLL = 8.99e-08)
LL=13919.2  AIC=-27790.4  BIC=-27638.2
  Fitting 5-state HMM …   Reached max iterations (300).
LL=13969.8  AIC=-27869.7  BIC=-27647.8

 N States  Train LL  Test LL      AIC      BIC  # Params
        2   13609.3   3327.2 -27202.6 -27151.9         8
        3   13865.4   3364.4 -27700.9 -27605.8        15
        4   13919.2   3382.9 -27790.4 -27638.2        24
        5   13969.8   3396.1 -27869.7 -27647.8        35


In [27]:
from utils.viz_utils import plot_aic_bic

model_names = [f"{r['N States']}-state" for r in results]
aics = [r["AIC"] for r in results]
bics = [r["BIC"] for r in results]

fig = plot_aic_bic(model_names, aics, bics, title="HMM Model Selection: AIC vs BIC")
fig.show()

## 9. Out-of-Sample Evaluation

In [28]:
df_test = df_test.copy()
df_test["HMM_State"] = hmm3.predict(X_test)

fig = plot_price_with_regimes(df_test, "HMM_State",
                               title="SPY Test Set - HMM Regime Detection")
fig.show()

stats_test = regime_statistics(df_test["Returns"].values, df_test["HMM_State"].values)
print("\nOut-of-sample regime statistics:")
print(stats_test.round(4).to_string(index=False))


Out-of-sample regime statistics:
 State  Mean Return (ann.)  Volatility (ann.)  Sharpe (ann.)  Skewness  Excess Kurtosis  # Days  % of Sample
     0              0.3530             0.0830         4.2516    0.0029           0.0351     427      40.7443
     1              0.0586             0.1877         0.3124   -0.0273           0.0420     587      56.0115
     2             -1.6210             0.5064        -3.2011    1.0104           2.1909      34       3.2443


## 10. Summary

| Component | Implementation |
|---|---|
| Gaussian emission $b_j(O_t)$ | `hmm_core._emission_matrix` |
| Forward algorithm (scaled) | `hmm_core.forward_pass` |
| Backward algorithm (scaled) | `hmm_core.backward_pass` |
| State occupancy $\gamma_t(i)$ | `hmm_core.compute_gamma` |
| Transition occupancy $\xi_t(i,j)$ | `hmm_core.compute_xi` |
| Viterbi decoding | `hmm_core.viterbi` |
| Baum-Welch EM | `GaussianHMM.fit` |
| Model selection (AIC/BIC) | `utils.metrics` |

**Next:** `02_hmm_libraries.ipynb` — reproduce the same results using `hmmlearn` and `pomegranate`.